### 1.0 Libs

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import statsmodels.api as sm
import inflection
import warnings
warnings.filterwarnings('ignore')
from matplotlib import pyplot as plt
from scipy.optimize import curve_fit
from statsmodels.base.model import GenericLikelihoodModel

### 1.1 Functions

In [ ]:
def gauss(x, a, mu, sigma):
    return a * np.exp(-((x - mu)**2) / (2 * sigma**2))

def elasticity_gauss(P, mu, sigma):
    return -(P * (P - mu)) / (sigma**2)

### 1.2 Loading data

In [ ]:
df_raw = pd.read_csv('../datasets/df_ready.csv', encoding='unicode_escape', low_memory=False)

### 2.0 Descrição dos dados

In [ ]:
df1 = df_raw.copy()

### 2.1 Renomear colunas

In [ ]:
cols_old = ['Unnamed: 0', 'Date_imp', 'Date_imp_d', 'Cluster', 'Category_name', 'name', 'price', 'disc_price', 
       'merchant', 'condition', 'Disc_percentage', 'isSale', 'Imp_count', 'brand', 'p_description',
       'currency', 'dateAdded', 'dateSeen', 'dateUpdated', 'imageURLs', 'manufacturer', 'shipping', 'sourceURLs', 
       'weight', 'Date_imp_d.1', 'Day_n', 'month', 'month_n', 'day', 'Week_Number', 'Zscore_1', 'price_std']

snakecase = lambda x: inflection.underscore(x)
cols_new = list(map(snakecase, cols_old))
df1.columns = cols_new

### 2.2 Change types

In [ ]:
df1['date_imp'] = pd.to_datetime(df1['date_imp'])
df1['date_added'] = pd.to_datetime(df1['date_added'])
#df1['date_seen'] = pd.to_datetime(df1['date_seen'])
df1['date_updated'] = pd.to_datetime(df1['date_updated'])

### 2.3 Seleção de colunas

In [ ]:
cols_drop = ['unnamed: 0', 'cluster', 'imp_count', 'zscore_1', 'price_std', 'date_imp_d', 'condition', 'is_sale',
            'currency', 'date_added', 'date_seen', 'date_updated', 'image_ur_ls', 'shipping', 'source_ur_ls', 
            'weight', 'date_imp_d.1']
df1 = df1.drop(cols_drop, axis=1)

### 3.0 Análise dos dados

In [ ]:
df2 = df1.copy()

### escolha da categoria e do produto:

In [ ]:
aux = df2[(df2['category_name'] == 'speaker, portable, bluetooth') & (df2['name'] == 'JBL Clip2 Portable Speaker')]

plt.figure(figsize=(15, 8))
sns.scatterplot(x='date_imp', y='price', data=aux, hue='merchant')
plt.xticks(rotation=90, fontsize=7)
plt.title('Preço em função da data para JBL Clip2 Portable Speaker')
plt.show;

### escolha do merchant: 

In [ ]:
aux1 = df2[(df2['category_name'] == 'speaker, portable, bluetooth') & (df2['merchant'] == 'Bestbuy.com') & (df2['name'] == 'JBL Clip2 Portable Speaker')]

### Determinando a demanda (ou quantidade vendida):

In [ ]:
df_demanda= aux1.groupby('price')['name'].count().reset_index(name='quantidade')

plt.scatter(df_demanda['price'], df_demanda['quantidade'])
plt.xlabel("Preço")
plt.ylabel("Quantidade")
plt.title("Curva de Demanda: Preço vs Quantidade")
plt.show()

### Determinando preço ótimo empírico:

In [ ]:
df_demanda['receita'] = df_demanda['price'] * df_demanda['quantidade']
print(df_demanda.loc[df_demanda['receita'].idxmax()])

plt.figure(figsize=(10, 4))
ax = sns.barplot(x='price', y='receita', data=df_demanda, palette='viridis')
plt.xlabel('Preço')
plt.ylabel('Receita')
plt.xticks(rotation=45, fontsize=8)
plt.title('Receita referente a cada (preço * quantidade)')

for i, (receita, quantidade) in enumerate(zip(df_demanda['receita'], df_demanda['quantidade'])):
    ax.text(i, receita + 5, str(quantidade), ha='center', va='bottom', fontsize=8)  

plt.show()

### Se quantidade de agrupamentos for menor que 4, calcular a elasticidade local:

In [ ]:
# encontrar o preço ótimo (máxima receita)
idx_max = df_demanda['receita'].idxmax()
preco_otimo = df_demanda.loc[idx_max, 'price']
q_otimo = df_demanda.loc[idx_max, 'quantidade']

# pegar vizinhos (se existirem)
if idx_max > 0 and idx_max < len(df_demanda) - 1:
    preco_menor, q_menor = df_demanda.loc[idx_max - 1, ['price', 'quantidade']]
    preco_maior, q_maior = df_demanda.loc[idx_max + 1, ['price', 'quantidade']]

    delta_p = preco_maior - preco_menor
    delta_q = q_maior - q_menor

    # elasticidade em torno do preço ótimo
    E = (delta_q / q_otimo) / (delta_p / preco_otimo)

    print(f"Preço ótimo: {preco_otimo}, Quantidade: {q_otimo}")
    print(f"Elasticidade em torno de {preco_otimo}: {E:.2f}")
else:
    print("Não há vizinhos suficientes para calcular a elasticidade.")
    print("É necessário mais agrupamentos de preços para fazer esse cálculo.")

### Se quantidade de agrupamentos for maior que 4, fazer os ajustes linear e log-log:

In [ ]:
# Modelo Linear: Q = a + B*P
X_lin = sm.add_constant(df_demanda['price'])
y_lin = df_demanda['quantidade']

model_lin = sm.OLS(y_lin, X_lin).fit()
print("\n=== Modelo Linear ===")
print(model_lin.summary())

# Modelo Log-Log: ln(Q) = a + B ln(P)
df_demanda = df_demanda[(df_demanda['quantidade'] > 0) & (df_demanda['price'] > 0)]  # garantir que não há log de zero/negativo
X_log = sm.add_constant(np.log(df_demanda['price']))
y_log = np.log(df_demanda['quantidade'])

model_log = sm.OLS(y_log, X_log).fit()
print("\n=== Modelo Log-Log ===")
print(model_log.summary())

# Coeficientes:
beta_lin = model_lin.params['price']
beta_log = model_log.params['price']

print("\nCoeficiente do modelo linear:", beta_lin)
print("Elasticidade no modelo linear (no preço médio):", beta_lin * (df_demanda['price'].mean() / df_demanda['quantidade'].mean()))
print("Elasticidade no modelo log-log:", beta_log)

### Se as métricas escolhidas estiverem dentro dos valores 'significativos', fazer os gráficos dos ajustes:
### métricas: p-valor < 0.05
###           r^2 > 0.3
###           prob f-statistic < 0.05
### o ajuste apresentado graficamente é aquele que possui os melhores valores das métricas

### Ajuste linear:

In [ ]:
# coeficientes da regressão
beta0, beta1 = model_lin.params

precos = np.linspace(df_demanda['price'].min(), df_demanda['price'].max(), 200)
quantidades_pred = beta0 + beta1 * precos
receita = precos * quantidades_pred
elasticidade = beta1 * (precos / quantidades_pred)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Demanda
axes[0].scatter(df_demanda['price'], df_demanda['quantidade'], label="Dados", color="black")
axes[0].plot(precos, quantidades_pred, color="blue", label="Ajuste linear")
axes[0].set_title("Demanda vs Preço")
axes[0].set_xlabel("Preço")
axes[0].set_ylabel("Quantidade")
axes[0].legend()
axes[0].grid(alpha=0.3)

# 2. Receita
axes[1].plot(precos, receita, color="blue", label="Receita estimada")
axes[1].set_title("Receita vs Preço")
axes[1].set_xlabel("Preço")
axes[1].set_ylabel("Receita")
axes[1].legend()
axes[1].grid(alpha=0.3)

# 3. Elasticidade
axes[2].plot(precos, elasticidade, color="blue")
axes[2].set_title("Elasticidade vs Preço")
axes[2].set_xlabel("Preço")
axes[2].set_ylabel("Elasticidade")
axes[2].legend()
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

### Ajuste log-log:

In [ ]:
# coeficientes da regressão
beta0, beta1 = model_log.params 

precos = np.linspace(df_log['price'].min(), df_log['price'].max(), 200)
quantidades_pred = np.exp(beta0 + beta1 * np.log(precos))
receita = precos * quantidades_pred
elasticidade = np.repeat(beta1, len(precos))  # constante no modelo log-log

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Demanda
axes[0].scatter(df_log['price'], df_log['quantidade'], label="Dados", color="black")
axes[0].plot(precos, quantidades_pred, color="blue", label="Ajuste log-log")
axes[0].set_title("Demanda (log-log)")
axes[0].set_xlabel("Preço")
axes[0].set_ylabel("Quantidade")
axes[0].legend()
axes[0].grid(alpha=0.3)

# 2. Receita
axes[1].plot(precos, receita, color="blue", label="Receita estimada")
axes[1].set_title("Receita vs Preço (log-log)")
axes[1].set_xlabel("Preço")
axes[1].set_ylabel("Receita")
axes[1].legend()
axes[1].grid(alpha=0.3)

# 3. Elasticidade
axes[2].plot(precos, elasticidade, color="blue", label=f"Elasticidade = {beta1:.2f}")
axes[2].axhline(-1, color="red", linestyle="--", label="Elasticidade = -1")
axes[2].set_title("Elasticidade vs Preço (log-log)")
axes[2].set_xlabel("Preço")
axes[2].set_ylabel("Elasticidade")
axes[2].legend()
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

### Se as métricas dos ajustes lineares não estiverem dentro da faixa estabelecida, fazer o ajuste 
### não linear. O modelo escolhido é uma gaussiana.

In [ ]:
preco = df_demanda['price'].values
quantidade = df_demanda['quantidade'].values

# curve_fit
p0 = [quantidade.max(), np.median(preco), np.std(preco)]
popt, pcov = curve_fit(gauss, preco, quantidade, p0=p0, maxfev=10000)
a_cf, mu_cf, sigma_cf = popt
se_cf = np.sqrt(np.diag(pcov))

# statsmodels (GenericLikelihoodModel)
class GaussModel(GenericLikelihoodModel):
    def nloglikeobs(self, params):
        a, mu, sigma = params
        yhat = gauss(preco, a, mu, sigma)
        resid = quantidade - yhat
        sigma_res = np.std(resid)
        return 0.5 * np.log(2*np.pi) + np.log(sigma_res) + 0.5*(resid/sigma_res)**2

mod = GaussModel(quantidade)
mod.endog_index = preco
res = mod.fit(start_params=p0, method="bfgs", disp=False, maxiter=10000)
a_sm, mu_sm, sigma_sm = res.params

# elasticidade
df_demanda['elasticidade_curvefit'] = elasticity_gauss(df_demanda['price'], mu_cf, sigma_cf)
df_demanda['elasticidade_statsmodels'] = elasticity_gauss(df_demanda['price'], mu_sm, sigma_sm)

print("curve_fit params:", popt, "std err:", se_cf)
print(res.summary())
print(df_demanda)

### Apresentação gráfica do ajuste gaussiano:

In [ ]:
# Parâmetros estimados
a_cf, mu_cf, sigma_cf = popt  # do curve_fit
a_sm, mu_sm, sigma_sm = res.params  # do statsmodels

# Grid de preços para curvas suaves
preco_grid = np.linspace(df_demanda['price'].min(), df_demanda['price'].max(), 300)

# Previsões
q_pred_cf = gauss(preco_grid, a_cf, mu_cf, sigma_cf)
q_pred_sm = gauss(preco_grid, a_sm, mu_sm, sigma_sm)

# 1. Curva ajustada vs dados
plt.figure(figsize=(10,4))
plt.scatter(df_demanda['price'], df_demanda['quantidade'], color="blue", label="Observado")
plt.plot(preco_grid, q_pred_cf, color="red", linewidth=2, label="Gaussiano (curve_fit)")
plt.plot(preco_grid, q_pred_sm, color="orange", linestyle="--", linewidth=2, label="Gaussiano (statsmodels)")
plt.axvline(mu_cf, color="red", linestyle=":", label=f"μ cf ≈ {mu_cf:.0f}")
plt.axvline(mu_sm, color="orange", linestyle=":", label=f"μ sm ≈ {mu_sm:.0f}")
plt.title("Demanda Observada vs Curvas Gaussianas Ajustadas")
plt.xlabel("Preço")
plt.ylabel("Quantidade")
plt.legend()
plt.show()

# 2. Resíduos
q_fit_cf = gauss(df_demanda['price'], a_cf, mu_cf, sigma_cf)
q_fit_sm = gauss(df_demanda['price'], a_sm, mu_sm, sigma_sm)
residuos_cf = df_demanda['quantidade'] - q_fit_cf
residuos_sm = df_demanda['quantidade'] - q_fit_sm

plt.figure(figsize=(10,4))
plt.axhline(0, color="black", linestyle="--")
plt.scatter(df_demanda['price'], residuos_cf, color="red", label="Resíduos curve_fit")
plt.scatter(df_demanda['price'], residuos_sm, color="orange", marker="x", label="Resíduos statsmodels")
plt.title("Resíduos dos Ajustes Gaussianos")
plt.xlabel("Preço")
plt.ylabel("Resíduo (Obs - Ajustado)")
plt.legend()
plt.show()

# 3. Elasticidade vs preço
elasticidade_cf = elasticity_gauss(preco_grid, mu_cf, sigma_cf)
elasticidade_sm = elasticity_gauss(preco_grid, mu_sm, sigma_sm)

plt.figure(figsize=(10,4))
plt.plot(preco_grid, elasticidade_cf, color="red", linewidth=2, label="Elasticidade (curve_fit)")
plt.plot(preco_grid, elasticidade_sm, color="orange", linestyle="--", linewidth=2, label="Elasticidade (statsmodels)")
plt.axhline(0, color="black", linestyle="--")
plt.axvline(mu_cf, color="red", linestyle=":", label=f"μ cf ≈ {mu_cf:.0f}")
plt.axvline(mu_sm, color="orange", linestyle=":", label=f"μ sm ≈ {mu_sm:.0f}")
plt.title("Elasticidade-Preço da Demanda (Modelo Gaussiano)")
plt.xlabel("Preço")
plt.ylabel("Elasticidade")
plt.legend()
plt.show()

### Se a regressão não linear gaussiana retornar resultados estatisticamente significativos, mas com valores
### de elasticidades muito altos (E > 10), os ajustes não lineares não serão considerados para o cálculo 
### da elasticidade. O cálculo da elasticidade será feito localmente.

### O mesmo acontecerá se o ajuste não retornar resultados estatisticamente significativos.
### p-valor do coeficiente par1 > 0.05

In [ ]:
# encontrar o preço ótimo (máxima receita)
idx_max = df_demanda['receita'].idxmax()
preco_otimo = df_demanda.loc[idx_max, 'price']
q_otimo = df_demanda.loc[idx_max, 'quantidade']

# pegar vizinhos (se existirem)
if idx_max > 0 and idx_max < len(df_demanda) - 1:
    preco_menor, q_menor = df_demanda.loc[idx_max - 1, ['price', 'quantidade']]
    preco_maior, q_maior = df_demanda.loc[idx_max + 1, ['price', 'quantidade']]

    delta_p = preco_maior - preco_menor
    delta_q = q_maior - q_menor

    # elasticidade em torno do preço ótimo
    E = (delta_q / q_otimo) / (delta_p / preco_otimo)

    print(f"Preço ótimo: {preco_otimo}, Quantidade: {q_otimo}")
    print(f"Elasticidade em torno de {preco_otimo}: {E:.2f}")
else:
    print("Não há vizinhos suficientes para calcular a elasticidade.")
    print("É necessário mais agrupamentos de preços para fazer esse cálculo.")